# 31 — Historical compact soft MoE, three-seed confirmation

Retrains the exact architecture and optimization contract of `nfv3_4way_moe_soft_comparable_seed0_v1` on seeds 0, 1, and 2: shared `47→128→64` encoder, four linear `64→22` dataset experts, linear dense gate, Stage A/B/C = `30/10/30`, task-only gate loss, and `0.1` load balancing. Each seed controls model initialization, data splitting, epoch shuffling, latent sampling, and deterministic CUDA execution. Validation macro-F1 selects the Stage-C checkpoint; test data is evaluated only after the configuration is locked. The notebook also produces the current metric suite, resource accounting, training curves, and A/B/C latent reports with PCA, UMAP, probes, and class-focus figures.


In [ ]:
# ======================= EDIT THIS CELL ONLY =======================
GITHUB_OWNER = 'selimsidan'
GITHUB_REPO = 'dataset_moe_nids'
GITHUB_BRANCH = 'main'
GITHUB_SECRET_NAME = 'GITHUB_TOKEN'
DRIVE_DATA_DIR = '/content/drive/MyDrive/NIDS_datasets'
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/NIDS_analysis_outputs/dataset_moe_nids_runs'
STUDY_CONFIG = 'config/compact_soft_moe_3seed.yaml'
PREFIX = 'nfv3_4way_moe_soft_comparable_repro_v1'
HISTORICAL_RUN = 'nfv3_4way_moe_soft_comparable_seed0_v1'
EXECUTE = False  # First run the dry-run; then change this to True.
MAX_NEW_SEEDS = 3  # Use 1 for one resumable seed per Colab session.
RUN_TESTS = True
# ==================================================================


## Secure checkout and environment setup


In [ ]:
import base64, json, os, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata
drive.mount('/content/drive')
token = userdata.get(GITHUB_SECRET_NAME)
if not token:
    raise RuntimeError(f'Add {GITHUB_SECRET_NAME} in Colab Secrets and grant access.')
auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
git_env = os.environ | {
    'GIT_CONFIG_COUNT': '1',
    'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
    'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {auth}',
}
repo = Path('/content') / GITHUB_REPO
url = f'https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git'
if (repo / '.git').is_dir():
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', 'origin', GITHUB_BRANCH], env=git_env, check=True)
else:
    subprocess.run(['git', 'clone', '--branch', GITHUB_BRANCH, '--single-branch', url, str(repo)], env=git_env, check=True)
git_env.clear(); token = auth = None
os.chdir(repo)
os.environ['NIDS_DRIVE_BASE'] = DRIVE_DATA_DIR
os.environ['NIDS_OUTPUT_DIR'] = DRIVE_OUTPUT_DIR
os.environ['NIDS_SCRATCH_DIR'] = '/content/dataset_moe_nids_scratch'
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
print('Repository commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())


## Contract preflight, dry-run, and resumable execution

The dry-run prints all three seed commands without training. Execution refuses a model that is not exactly 47 inputs, 22 classes, and 20,380 deployed parameters. Existing compatible checkpoints are resumed; a changed contract requires a new prefix.


In [ ]:
import torch, yaml
from training.logging_utils import run_streaming_logged
summary_dir = Path(DRIVE_OUTPUT_DIR) / 'results' / f'{PREFIX}_summary'
summary_dir.mkdir(parents=True, exist_ok=True)
study = yaml.safe_load(Path(STUDY_CONFIG).read_text())
locked = study['backbone']
print('Locked architecture:', locked['architecture'])
print('Seeds:', study['seeds'])
print('Encoder:', [47, *locked['model']['encoder']['hidden_dims'], locked['model']['latent_dim']])
print('Experts: 4 ×', [locked['model']['latent_dim'], *locked['model']['expert']['hidden_dims'], 22])
print('Expected deployed parameters:', f"{study['expected_total_parameters']:,}")
print('Historical reference:', study['historical_reference_run'])
mode = 'TRAINING ENABLED' if EXECUTE else 'DRY RUN — NO TRAINING'
print('=' * 80, mode, '=' * 80, sep='\n', flush=True)
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before executing the study.')
if RUN_TESTS:
    subprocess.run([
        sys.executable, '-m', 'pytest', '-q',
        'tests/test_compact_soft_moe_run.py', 'tests/test_latent_space.py',
        'tests/test_out_of_core.py', 'tests/test_training_observability.py',
    ], check=True)
command = [
    sys.executable, '-u', '-m', 'training.compact_soft_moe_run',
    '--study-config', STUDY_CONFIG, '--prefix', PREFIX,
]
print('=== Complete dry-run graph ===', flush=True)
run_streaming_logged(
    command, str(summary_dir / 'Notebook_Dry_Run.log'),
    env=os.environ.copy(), heartbeat_seconds=30, label='notebook31-dry-run',
)
if EXECUTE:
    launch = [*command, '--execute', '--max-new-seeds', str(MAX_NEW_SEEDS)]
    stream_log = summary_dir / 'Notebook_Training_Stream.log'
    print('=== Executing fixed three-seed confirmation with enforced live output ===', flush=True)
    print('Persistent aggregate log:', stream_log, flush=True)
    run_streaming_logged(
        launch, str(stream_log), env=os.environ.copy(),
        heartbeat_seconds=30, label='notebook31-training',
    )
else:
    print('NO TRAINING WAS STARTED. Review the graph, set EXECUTE=True, and rerun this cell.', flush=True)


## Three-seed metrics, guardrails, and historical comparison


In [ ]:
import pandas as pd
from IPython.display import Image, display
manifest_path = summary_dir / 'Final_Study_Manifest.json'
if not manifest_path.is_file():
    print('The three-seed confirmation is not complete yet:', summary_dir)
else:
    manifest = json.loads(manifest_path.read_text())
    print(json.dumps(manifest, indent=2))
    for filename in [
        'Three_Seed_Overall_Summary.csv',
        'Three_Seed_Guardrails.csv',
        'Three_Seed_Per_Dataset_Summary.csv',
        'Three_Seed_Per_Class_Summary.csv',
        'Three_Seed_Resource_Accounting.csv',
        'Three_Seed_Selected_Epochs.csv',
    ]:
        print('===', filename, '===')
        display(pd.read_csv(summary_dir / filename))
    legacy_path = Path(DRIVE_OUTPUT_DIR) / 'results' / HISTORICAL_RUN / 'Overall_Metrics.csv'
    if legacy_path.is_file():
        legacy = pd.read_csv(legacy_path)
        legacy = legacy[legacy['origin'] == 'ALL'].copy()
        legacy.insert(0, 'source', 'historical_seed0')
        reproduced = pd.read_csv(summary_dir / 'Three_Seed_Overall.csv')
        reproduced.insert(0, 'source', reproduced['seed'].map(lambda value: f'reproduction_seed{value}'))
        common = ['source', *[name for name in (
            'seed', 'accuracy', 'balanced_accuracy', 'macro_precision',
            'macro_recall', 'macro_f1', 'weighted_f1',
            'roc_auc_ovr_macro', 'pr_auc_ovr_macro',
        ) if name in legacy.columns and name in reproduced.columns]]
        print('=== Historical seed 0 versus the three fresh runs (common metrics) ===')
        display(pd.concat([legacy.reindex(columns=common), reproduced.reindex(columns=common)], ignore_index=True))
    seed0_dir = Path(manifest['result_dirs']['0'])
    for filename in ['Gate_By_Dataset.csv', 'Expert_Utilization.csv', 'Expert_Performance_By_Dataset.csv']:
        path = seed0_dir / filename
        if path.is_file():
            print('=== Representative seed-0', filename, '===')
            display(pd.read_csv(path))


## Training logs, epoch histories, and learning curves


In [ ]:
if manifest_path.is_file():
    print('Aggregate live-training log:', summary_dir / 'Notebook_Training_Stream.log')
    for filename in ['Training_History_Summary_3Seed.csv', 'Training_Log_Manifest_3Seed.csv']:
        print('===', filename, '===')
        display(pd.read_csv(summary_dir / filename))
    print('=== Cross-seed training curves ===')
    for path in sorted((summary_dir / 'training_figures').glob('*.png')):
        print(path.name)
        display(Image(filename=str(path)))
    seed0_training = Path(manifest['result_dirs']['0']) / 'training'
    print('=== Representative seed-0 learning curves ===')
    for path in sorted((seed0_training / 'training_figures').glob('*.png')):
        print(path.name)
        display(Image(filename=str(path)))
    print('Per-stage persistent log manifest:')
    display(pd.read_csv(seed0_training / 'Training_Log_Manifest.csv'))
else:
    print('Training summaries will appear after all three seeds complete. Live output is saved under:', summary_dir)


## A/B/C latent-space metrics and saved figures

For this shared-encoder architecture, Stage B does not create a new latent representation: its `gate` snapshot is an explicit alias of the Stage-A encoder. The meaningful geometric change is therefore Stage A → jointly fine-tuned Stage C, while Stage-B expert competence is measured through the task and expert-performance reports.


In [ ]:
if manifest_path.is_file():
    for filename in [
        'Latent_Snapshot_Summary_3Seed.csv',
        'Latent_Probe_Summary_3Seed.csv',
        'Latent_Per_Class_Summary_3Seed.csv',
    ]:
        print('===', filename, '===')
        display(pd.read_csv(summary_dir / filename))
    for path in sorted((summary_dir / 'latent_figures').glob('*.png')):
        print(path.name)
        display(Image(filename=str(path)))
    seed0 = Path(manifest['result_dirs']['0']) / 'latent' / 'cumulative' / 'latent_figures'
    representative = [
        seed0 / 'A__shared_initialization__umap.png',
        seed0 / 'B__gate__umap.png',
        seed0 / 'C__gate__umap.png',
    ]
    print('=== Representative seed-0 UMAP gallery ===')
    for path in representative:
        if path.is_file():
            print(path.name)
            display(Image(filename=str(path)))
    print('All per-seed PCA, UMAP, class-focus, metric, manifest, and embedding files are under:')
    for seed, path in manifest['result_dirs'].items():
        print(f'  seed {seed}:', Path(path) / 'latent')
